<a href="https://colab.research.google.com/github/fasih779/deep-learning/blob/master/simple%20chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import zipfile
with zipfile.ZipFile("/content/test.csv.zip", 'r') as zip_ref:
    zip_ref.extractall("./dataset")

In [19]:
import pandas as pd


In [20]:
df=pd.read_csv('/content/dataset/test.csv')

In [21]:
df.head(1)

,personas,additional_context,previous_utterance,context,free_messages,guided_messages,suggestions,guided_chosen_suggestions,label_candidates
0,['i hate talking to people.' 'i believe dragon...,Social anxiety,"['Wow, I am never shy. Do you have anxiety?'\n...",wizard_of_wikipedia,['and why is that?'\n 'interesting but I know ...,"[""I think it's because in my head, I think eve...","{'convai2': array([""i've no idea i am also ver...",['wizard_of_wikipedia' '' ''],[array(['Oh nice! My brother in law is a lawye...


In [22]:
df.drop(['context','guided_messages','suggestions','guided_chosen_suggestions','label_candidates'],inplace=True,axis=1)

In [23]:
df.sample(1)

,personas,additional_context,previous_utterance,free_messages
7,['i like going to the moves.' 'i work in a cir...,NaN,['I fell bad about myself because I voted on T...,['If Trump was a giant tree I would love cut i...


In [64]:
df['free_messages'] = df['free_messages'].apply(lambda x: "<start> " + str(x) + " <end>")

In [24]:
df['additional_context']=df['additional_context'].fillna('other')

In [65]:

df.head(2)

,personas,additional_context,previous_utterance,free_messages,encoder_input
0,i hate talking to people i believe dragons are...,social anxiety,wow i am never shy do you have anxiety\n yes i...,<start> and why is that\n interesting but i kn...,[PERSONA] i hate talking to people i believe d...
1,i have three daughters my wife and i like to g...,other,my turtle ran away from me today\n oh my god d...,<start> thats funny no i let him roam around t...,[PERSONA] i have three daughters my wife and i...


In [40]:
import re


In [41]:
df['free_messages']=df['free_messages'].apply(lambda x:re.sub(r'[^\w\s]','',x))


In [48]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.lower()
        df[col] = df[col].str.replace(r'[^\w\s]', '', regex=True)
        df[col] = df[col].str.strip()

In [57]:
df['encoder_input'] = (
    '[PERSONA] ' + df['personas'] + ' ' +
    '[PREV] ' + df['previous_utterance'] + ' ' +
    '[CONTEXT] ' + df['additional_context']
)

In [58]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [59]:
encoder_texts = df['encoder_input'].tolist()


In [60]:
encoder_tokenizer = Tokenizer(filters='')  # we already cleaned punctuation
encoder_tokenizer.fit_on_texts(encoder_texts)

# Convert text to sequences
encoder_seq = encoder_tokenizer.texts_to_sequences(encoder_texts)

# Pad sequences to same length
max_encoder_len = max(len(seq) for seq in encoder_seq)  # or set a fixed length
encoder_seq = pad_sequences(encoder_seq, maxlen=max_encoder_len, padding='post')

print("Vocabulary size:", len(encoder_tokenizer.word_index)+1)
print("Encoder sequences shape:", encoder_seq.shape)

Vocabulary size: 5310
Encoder sequences shape: (980, 112)


In [67]:
decoder_texts = df['free_messages'].tolist()  # should include <start> and <end>

# Tokenizer
decoder_tokenizer = Tokenizer(filters='')
decoder_tokenizer.fit_on_texts(decoder_texts)

decoder_seq = decoder_tokenizer.texts_to_sequences(decoder_texts)
max_decoder_len = max(len(seq) for seq in decoder_seq)
decoder_seq = pad_sequences(decoder_seq, maxlen=max_decoder_len, padding='post')

import numpy as np
decoder_target = np.zeros_like(decoder_seq)
decoder_target[:, :-1] = decoder_seq[:, 1:]

In [68]:
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, Attention
from tensorflow.keras.models import Model

In [69]:

encoder_vocab_size = len(encoder_tokenizer.word_index) + 1
decoder_vocab_size = len(decoder_tokenizer.word_index) + 1


max_encoder_len = encoder_seq.shape[1]
max_decoder_len = decoder_seq.shape[1]


embedding_dim = 256
latent_dim = 512

In [70]:

encoder_inputs = Input(shape=(max_encoder_len,), name="encoder_inputs")

encoder_embedding = Embedding(input_dim=encoder_vocab_size, output_dim=embedding_dim, mask_zero=True)(encoder_inputs

encoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

encoder_states = [state_h, state_c]

In [71]:

decoder_inputs = Input(shape=(max_decoder_len,), name="decoder_inputs")


decoder_embedding = Embedding(input_dim=decoder_vocab_size, output_dim=embedding_dim, mask_zero=True)(decoder_inputs)

# LSTM
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

In [72]:
# Dot-product attention
attention = tf.keras.layers.Attention(name="attention_layer")
context_vector = attention([decoder_outputs, encoder_outputs])

# Concatenate context with decoder output
decoder_concat_input = tf.keras.layers.Concatenate(axis=-1)([decoder_outputs, context_vector])

In [73]:
decoder_dense = Dense(decoder_vocab_size, activation='softmax', name="decoder_dense")
decoder_outputs_final = decoder_dense(decoder_concat_input)

In [74]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs_final)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 112)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 162)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 112, 256)  │  1,359,360 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 112)       │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 162, 256)  │  1,952,256 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 112,      │  1,574,912 │ embedding[0][0],  │
│                     │ 512), (None,      │            │ not_equal[0][0]   │
│                     │ 512), (None,      │            │                   │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 162,      │  1,574,912 │ embedding_1[0][0… │
│                     │ 512), (None,      │            │ encoder_lstm[0][… │
│                     │ 512), (None,      │            │ encoder_lstm[0][… │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_layer     │ (None, 162, 512)  │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 162, 1024) │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ attention_layer[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, 162, 7626) │  7,816,650 │ concatenate[0][0] │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 14,278,090 (54.47 MB)

 Trainable params: 14,278,090 (54.47 MB)

 Non-trainable params: 0 (0.00 B)

In [76]:
decoder_target_input = decoder_target[..., np.newaxis]
history = model.fit(
    [encoder_seq, decoder_seq],
    decoder_target_input,
    batch_size=64,
    epochs=50,
    validation_split=0.1
)

Epoch 1/50
 8/14 ━━━━━━━━━━━━━━━━━━━━ 2:04 21s/step - accuracy: 0.0359 - loss: 8.7954

KeyboardInterrupt: 